#### Install Homebrew (for Linux / MacOs unix)

In [ ]:
# Run this cell
!/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

# Note: You can also run in the terminal but without the preceeding !
# /bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

In [2]:
# add homebrew to path and apply it to the current session
# For Linux
!echo 'eval "$(/home/linuxbrew/.linuxbrew/bin/brew shellenv)"'  >> ~/.bashrc
!eval "$(/home/linuxbrew/.linuxbrew/bin/brew shellenv)"

# For macOS M1/M2/M3 
# echo 'eval "$(/opt/homebrew/bin/brew shellenv)"' >> ~/.zprofile 
# eval "$(/opt/homebrew/bin/brew shellenv)"

# For masOS (Intel)
# echo 'eval "$(/usr/local/bin/brew shellenv)"'  >> ~/.zprofile
# eval "$(/usr/local/bin/brew shellenv)"

In [ ]:
# install homebrew dependencies
# For Debian or Ubuntu
!sudo apt-get install build-essential procps curl file git

# For Fedora
# sudo dnf group install development-tools
# sudo dnf install procps-ng curl file

# For CentOS Stream or RHEL
# sudo dnf group install 'Development Tools'
# sudo dnf install procps-ng curl file

# For Arch Linux
# sudo pacman -S base-devel procps-ng curl file git

### Installing Docker on Ubuntu Linux

In [ ]:
# uninstall old docker version if it already exists on machine.
!sudo apt remove docker docker-engine docker.io containerd runc docker-compose docker-compose-v2 docker-doc podman-docker

In [ ]:
# Update package index, install prerequisites and add Docker's official GPG key:
!sudo apt update
!sudo apt install ca-certificates curl
!sudo install -m 0755 -d /etc/apt/keyrings
!sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
!sudo chmod a+r /etc/apt/keyrings/docker.asc

In [ ]:
# Add the docker repository to Apt sources:
!echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "$VERSION_CODENAME") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null

In [ ]:
# Update apt package index, install Docker Engine, CLI, and plugins
!sudo apt update
!sudo apt install -y docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin

In [ ]:
# verify the installation was successful
!sudo docker run hello-world

### Install KitOps CLI

In [ ]:
!brew tap kitops-ml/kitops
!brew install kitops

In [ ]:
# check installation
!kit version

### Install the required Python libraries

In [ ]:
# we assume you have python already installed
!pip install kitops huggingface_hub

### Authenticate KitOps CLI with Jozu Hub

In [ ]:
# Run the below command in the terminal, you will be prompted to insert the username (email) and password, 
# you used to create the jozu hub account. 
# Note: As you are typing the password it won't show on most terminals so don't be surprised 
# just hit enter when you are done.


# kit login jozu.ml

### Download the model from hugging face


In [ ]:
from huggingface_hub import hf_hub_download

# Will download the quantized GGUF to the given local directory

# Download the Target model
hf_hub_download(
    repo_id="unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF", 
    filename="Qwen3-30B-A3B-Instruct-2507-UD-TQ1_0.gguf",
    local_dir="./Qwen3-family"
)

# Download the Draft gguf model
hf_hub_download(
    repo_id="unsloth/Qwen3-0.6B-GGUF", 
    filename="Qwen3-0.6B-Q8_0.gguf",
    local_dir="./Qwen3-family"
)

### Packaging & versioning as a ModelKit

In [ ]:
from kitops.modelkit.kitfile import Kitfile

# Create new Kitfile
kitfile = Kitfile()

# Set basic metadata
kitfile.manifestVersion = "1.0"
kitfile.package = {
    "name": "speculative-decoding-with-jozu",
    "version": "1.0",
    "description": "Kitfile for Target and Draft gguf models"
}


# # Configure model information
kitfile.model = {
    "name": "Qwen3-30B-A3B-Instruct-GGUF",
    "path": "Qwen3-family/Qwen3-30B-A3B-Instruct-2507-UD-TQ1_0.gguf", # path to the Target GGUF file.
    "version": "1.0",
    "license": "Apache 2.0",
    "description": "",
    "parts": [
        {
            "name": "Qwen3-0.6B-GGUF",
            "path": "Qwen3-family/Qwen3-0.6B-Q8_0.gguf"
        }
    ]
}

# You can also add other information like below:

# kitfile.code = [
#     {
#         "path": "...",
#         "description": "..."
#     }
# ]

# kitfile.datasets = [
#     {
#         "name": "dataset",
#         "path": "data/sample.csv",
#         "description": "full dataset",
#         "license": "Apache 2.0"
#     }
# ]

# kitfile.docs = [
#     {"path": "docs/README.md"},
#     {"path": "docs/LICENSE"}
# ]

# For more information on what you can add, see https://kitops.org/docs/pykitops/how-to-guides/


# You Can Save the Kitfile locally (Note: It is the Kitfile that specifies how the Modelkit will be bundled)
kitfile.save("Kitfile")


### Uploading the ModelKit to Jozu Hub

In [ ]:
from kitops.modelkit.manager import ModelKitManager, UserCredentials

# Configure the ModelKit manager
# Note: The email prefix of jack123@gmail.com is just jack123

modelkit_tag = "jozu.ml/email-prefix-here/speculative-decoding-with-jozu:latest" # all in lowercase
manager = ModelKitManager(
    working_directory=".",
    modelkit_tag=modelkit_tag,
    user_credentials=UserCredentials('full-email-here', 'password-here', namespace='speculative-decoding-with-jozu')
)
  
# Assign your Kitfile
manager.kitfile = kitfile

# Pack and push to Jozu Hub
manager.pack_and_push_modelkit(save_kitfile=True)

### Running the Model with Jozu Rapid Inference Container (RICs)

In [ ]:
# Run the below command on the terminal (replace placeholders with details)

# With speculative decoding.
# sudo docker run -it --rm \
#   -p 8000:8000 \
#   --gpus all \
#   --entrypoint /app/llama-server \
#   "jozu.ml/email-prefix-here/speculative-decoding-with-jozu/llama-cpp-cuda:latest" \
#   -m /Qwen3-family/Qwen3-30B-A3B-Instruct-2507-UD-TQ1_0.gguf \
#   -md /Qwen3-family/Qwen3-0.6B-Q8_0.gguf \
#   --jinja \
#   --chat-template-kwargs '{"enable_thinking": false}' \
#   --host 0.0.0.0 \
#   --port 8000 \
#   -c 40965 \
#   --draft-n 4


# Without speculative decoding.
# sudo docker run -it --rm \
#   -p 8000:8000 \
#   --gpus all \
#   --entrypoint /app/llama-server \
#   "jozu.ml/email-prefix-here/speculative-decoding-with-jozu/llama-cpp-cuda:latest" \
#   -m /Qwen3-family/Qwen3-30B-A3B-Instruct-2507-UD-TQ1_0.gguf \
#   --jinja \
#   --chat-template-kwargs '{"enable_thinking": false}' \
#   --host 0.0.0.0 \
#   --port 8000 \
#   -c 40965 


### Test Inference Speed

In [ ]:
# Run the below command on the terminal (It will report the generation speed for 2048 tokens across three runs).

# for i in 1 2 3; do
#   echo "================"
#   echo "Run $i result:"
#   curl -s http://localhost:8000/completion -d '{
#     "prompt": "Write an elaborate step by step technical guide on how to pre-train, finetune, and inference large language models.",
#     "n_predict": 2048,
#     "temperature": 0
#   }' | python3 -c "import json,sys; print(json.dumps(json.load(sys.stdin)['timings'], indent=2))"
#   echo "================"
# done